# M1: torsion + rigid-rotor baseline

A thin driver over the library. Run `00_smoke_test.ipynb` first, or install the wheel, so that `import riss_drucker` works.

Model (cm⁻¹): H = F p² + Σ_L V_L (1 − cos Lφ)/2 + A K² + B (J(J+1) − K²) + D_Kp K p, in the basis e^{imφ} |J K⟩.
Blocks are (J, K, σ = m mod n) plus cos/sin parity where it applies. Every block result carries an inertia-count certificate, a residual bound and an orthogonality check.

The parameters below are **illustrative** (methyl-top-like F, generic barriers). They are not fitted values for any molecule.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import riss_drucker as rd

WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
print(rd.__version__, rd.build_info()["git_commit"][:12], "simd:", rd.simd_level_active(), "linalg:", rd.linalg_backends())
print(rd.runtime_config().as_dict())

## 1. Compile once, inspect blocks

In [ ]:
model = rd.compile_torsion_rotor(n_fold=3, v_orders=[1, 2], m_max=60)
print(model.spec_json)
print("parameters:", model.param_names)
for b in model.blocks:
    print(f"{b['irrep']:>10}  dim={b['dim']:3d}  first basis fns: {b['basis_labels'][:4]}")
print(model.compile_certificates)

## 2. Barrier sweep: free rotor → harmonic oscillator

Lowest levels per species as V3 grows. The free-rotor limit is F m², and the large-barrier limit is n√(F V)(v + ½) minus the anharmonic correction. The A₁/E splitting of each torsional level shrinks as the barrier rises (tunnelling).

In [ ]:
F = 5.3
V3s = np.linspace(0.0, 1500.0, 61)
species = [b["irrep"] for b in model.blocks]
levels = {s: [] for s in species}
all_certified = True
for V in V3s:
    sol = rd.solve(model, {"F": F, "V3": V, "V6": 0.0}, nlowest=4)
    all_certified &= sol.certified
    for b, r in zip(sol.blocks, sol.results):
        levels[b["irrep"]].append(r["eigenvalues"][:4])
levels = {s: np.array(v) for s, v in levels.items()}
print("all blocks certified across the sweep:", all_certified)

def harmonic(v, V, n=3):  # Mathieu large-q expansion (A&S 20.2.30), q = V / (n^2 F)
    s, q = 2 * v + 1, V / (n * n * F)
    return n*np.sqrt(F*V)*(v+0.5) - n*n*F/4*((s*s+1)/8 + (s**3+3*s)/(128*np.sqrt(q)) + (5*s**4+34*s*s+9)/(4096*q))

print("V3 = 1500: E-species levels   ", np.round(levels["sigma=1"][-1][:3], 4))
print("           harmonic expansion ", np.round([harmonic(v, 1500.0) for v in range(3)], 4))
print("ground A1/E tunnelling splitting vs V3:",
      {int(V): float(levels["sigma=1"][i][0] - levels["sigma=0,+"][i][0]) for i, V in list(enumerate(V3s))[::15]})

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    styles = {"sigma=0,+": ("A1 (cos)", "-", "C0"), "sigma=0,-": ("A2 (sin)", "--", "C1"),
              "sigma=1": ("E", ":", "C2"), "sigma=2": (None, ":", "C2")}
    for s, (lab, ls, col) in styles.items():
        for j in range(4):
            ax.plot(V3s, levels[s][:, j], ls, color=col, lw=1.2, label=lab if j == 0 and lab else None)
    Vh = V3s[V3s > 200]
    for v in range(4):
        ax.plot(Vh, harmonic(v, Vh), color="0.6", lw=0.8, label="harmonic expansion" if v == 0 else None)
    ax.plot(V3s, V3s, color="k", lw=0.6, alpha=0.4, label="barrier top")
    ax.set(xlabel="V3 / cm$^{-1}$", ylabel="E / cm$^{-1}$", ylim=(0, 900), title=f"Torsional levels, F = {F} cm$^{{-1}}$")
    ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    out = WORK / "01_barrier_sweep.png"
    fig.savefig(out, dpi=150)
    print("saved", out)

## 3. Exact reference: Mathieu characteristic values (A species, V3 only)

In [ ]:
try:
    from scipy.special import mathieu_a, mathieu_b
except ImportError:
    mathieu_a = None
if mathieu_a is not None:
    m3 = rd.compile_torsion_rotor(n_fold=3, v_orders=[1], m_max=90)
    V = 400.0
    q = V / (9 * F)
    sol = rd.solve(m3, {"F": F, "V3": V}, nlowest=5, method="dense")
    a1 = next(r["eigenvalues"] for b, r in zip(sol.blocks, sol.results) if b["irrep"] == "sigma=0,+")
    ref = [V/2 + 9*F/4*mathieu_a(2*k, q) for k in range(5)]
    print("A1 levels :", np.round(a1, 9))
    print("Mathieu   :", np.round(ref, 9))
    print("max |diff|:", np.abs(np.array(a1) - ref).max())

## 4. Certificates are results, not comments

A starved Lanczos run is reported as uncertified, with the failing certificate named. It does not raise and is not silently accepted.

In [ ]:
big = rd.compile_torsion_rotor(n_fold=3, v_orders=[1], m_max=600)
ok = rd.solve(big, {"F": F, "V3": 400.0}, nlowest=10, method="lanczos")
bad = rd.solve(big, {"F": F, "V3": 400.0}, nlowest=40, method="lanczos", lanczos_max_basis=3, lanczos_max_runs=1)
print("normal run certified:", ok.certified, "| modes:", {r["lanczos_mode"] for r in ok.results})
print("starved run certified:", bad.certified)
for c in bad.failed_certificates()[:3]:
    print("  FAILED", c["block"], c["name"], c["detail"])

## 5. Torsion-rotation: K structure with K·p coupling

F(p − ρK)² = F p² − 2Fρ K p + Fρ² K², so D_Kp = −2Fρ, and A absorbs Fρ² (a gauge choice, see `docs/conventions.md`). Blocks (K, σ) and (−K, −σ) are degenerate.

In [ ]:
tr = rd.compile_torsion_rotor(n_fold=3, v_orders=[1], m_max=40, j_max=4, kp_coupling=True)
rho, A, B = 0.08, 3.0, 0.8
sol = rd.solve(tr, {"F": F, "V3": 400.0, "A": A + F * rho**2, "B": B, "D_Kp": -2 * F * rho}, nlowest=2, threads=4)
print("certified:", sol.certified, " blocks:", tr.num_blocks)
for row in [r for r in sol.levels() if r["J"] == 4 and r["index"] == 0][:9]:
    print(f"J=4 {row['sub']:>5} {row['irrep']:>10}  E = {row['energy']:10.4f}  ({row['assignment']}, w={row['weight']:.2f})")

In [ ]:
rec = rd.write_run_record(WORK / "run_record_m1.json", spec_hash=None, seeds={"solve": 0},
                          extra={"last_solve": sol.summary()})
print("run record:", rec)